In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from matplotlib import pyplot as plt
import os
import tensorflow_probability as tfp
tfd = tfp.distributions

tf.config.list_physical_devices('GPU')

2023-04-11 22:35:30.907721: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-04-11 22:35:30.977885: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-04-11 22:35:30.979017: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-04-11 22:35:32.376988: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-04-11 22:35:36.246512: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


[]

# importing data

In [2]:
df = pd.read_csv('../camels_info/camels_parameters.csv')
df

,Name,Omega_m,sigma_8,A_SN1,A_AGN1,A_SN2,A_AGN2,seed
0,LH_0,0.3090,0.9790,3.11234,1.12194,0.66850,0.53182,0
1,LH_1,0.3026,0.9394,3.42001,3.96137,1.03311,1.16070,1
2,LH_2,0.4282,0.7530,0.70613,0.37423,1.96292,0.62720,2
3,LH_3,0.1906,0.6286,1.60882,0.39887,0.86634,0.86634,3
4,LH_4,0.1382,0.6874,1.19914,0.27586,0.82188,0.91700,4
...,...,...,...,...,...,...,...,...
1092,CV_26,0.3000,0.8000,1.00000,1.00000,1.00000,1.00000,27
1093,EX_0,0.3000,0.8000,1.00000,1.00000,1.00000,1.00000,13560
1094,EX_1,0.3000,0.8000,1.00000,100.00000,1.00000,1.00000,13560
1095,EX_2,0.3000,0.8000,100.00000,1.00000,1.00000,1.00000,13560


In [3]:
out_dir = "../power_spectra/CO/20230313_no_std/"

def get_LH_files():
    fils = list(map(lambda fil: fil[:-4], \
               (filter(lambda fil: f"LH_" in fil, os.listdir(out_dir)))))
    sorter_1P_fils = lambda filname: int(filname.split('_')[-1].replace('n', '-'))
    return sorted(fils, key=sorter_1P_fils)

sim_names = get_LH_files()
sim_names[:5], sim_names[-5:], len(sim_names)

(['LH_0', 'LH_1', 'LH_2', 'LH_3', 'LH_4'],
 ['LH_995', 'LH_996', 'LH_997', 'LH_998', 'LH_999'],
 1000)

In [4]:
if "LH_603" in sim_names:
    sim_names.remove("LH_603")

In [5]:
non_nan_range = np.arange(24, 46)
all_curves = np.array([])
k_set = None
for fil in sim_names:
    with np.load(out_dir + fil + ".npz", allow_pickle=True) as data:
        curves = data['curves'].item()
        redshifts = data['redshifts'].item()
        ks = data['ks'].item()
    nan_sum = sum(sum(np.isnan(np.array(list(curves.values()))))[non_nan_range])
    assert (nan_sum == 0), fil
    assert (sum(np.isnan(np.array(list(curves.values()))))[23] == 34)
    assert (sum(np.isnan(np.array(list(curves.values()))))[46] == 34)
    if k_set is None:
        k_set = ks[0]
    for k in ks.values():
        assert((k == k_set).all()), print(fil, k, k_set)

In [6]:
num_samples = len(sim_names)
print(num_samples)
non_nan_range = np.arange(24, 46)
all_curves = np.zeros((num_samples, 34, 22)) + np.nan
all_cosmologies = np.zeros((num_samples, 6)) + np.nan
for ind, fil in enumerate(sim_names):
    with np.load(out_dir + fil + ".npz", allow_pickle=True) as data:
        curves = data['curves'].item()
        redshifts = data['redshifts'].item()
        ks = data['ks'].item()
    all_curves[ind] = np.array(list(curves.values()))[:, non_nan_range]
    all_cosmologies[ind] = df.loc[df['Name'] == fil].values[0][1:-1]

999


In [7]:
print(all_curves.shape, all_cosmologies.shape)
print(np.sum(np.isnan(all_curves)), np.sum(np.isnan(all_cosmologies)))

(999, 34, 22) (999, 6)
0 0


In [8]:
train_split, val_split, test_split = int(0.85*num_samples), \
            int(0.10*num_samples) + 1, int(0.05*num_samples) + 1

print(train_split, val_split, test_split, train_split+val_split+test_split)

train_x, val_x, test_x = np.split(all_curves, [train_split, train_split+val_split])
train_y, val_y, test_y = np.split(all_cosmologies, [train_split, train_split+val_split])

print(train_x.shape, val_x.shape, test_x.shape)
print(train_y.shape, val_y.shape, test_y.shape)

849 100 50 999
(849, 34, 22) (100, 34, 22) (50, 34, 22)
(849, 6) (100, 6) (50, 6)


# BNN

In [15]:
kl_divergence_function = (lambda q, p, _: tfd.kl_divergence(q, p) /  # pylint: disable=g-long-lambda
                           tf.cast(train_split, dtype=tf.float32))

In [26]:
input_shape = (34, 22, 1) 

model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=input_shape),
    tfp.layers.Convolution2DFlipout(filters = 4, kernel_size=(3, 3), padding='valid',
          kernel_divergence_fn=kl_divergence_function, activation=tf.nn.relu),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=None, padding="valid"),
    # tfp.layers.Convolution2DFlipout(filters = 4, kernel_size=(2, 2), padding='valid',
    #       kernel_divergence_fn=kl_divergence_function, activation=tf.nn.relu),
    # tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=None, padding="valid"),
    # tfp.layers.Convolution2DFlipout(filters = 4, kernel_size=(2, 2), padding='valid',
    #       kernel_divergence_fn=kl_divergence_function, activation=tf.nn.relu),
    # tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=None, padding="valid"),
    tf.keras.layers.Flatten(),
    tfp.layers.DenseFlipout(units = 32,
          kernel_divergence_fn=kl_divergence_function, activation=tf.nn.relu),
    tfp.layers.DenseFlipout(units = 32,
          kernel_divergence_fn=kl_divergence_function, activation=tf.nn.relu),
    tf.keras.layers.Dense(tfp.layers.MultivariateNormalTriL.params_size(6)),
    tfp.layers.MultivariateNormalTriL(event_size=6)
])

In [27]:
# Define loss
def negloglik(y_true, y_pred):
    return -tf.reduce_mean(y_pred.log_prob(y_true))
# Define the optimizer 
optimizer=tf.keras.optimizers.Adadelta(learning_rate=0.2, rho=0.98)
# optimizer=tf.keras.optimizers.Adadelta()
model.compile(optimizer, loss=['mse'], metrics=['mae', 'mse'], 
              experimental_run_tf_function=False)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_flipout (Conv2DFlipo  (None, 32, 20, 4)        76        
 ut)                                                             
                                                                 
 max_pooling2d (MaxPooling2D  (None, 16, 10, 4)        0         
 )                                                               
                                                                 
 flatten (Flatten)           (None, 640)               0         
                                                                 
 dense_flipout (DenseFlipout  (None, 32)               40992     
 )                                                               
                                                                 
 dense_flipout_1 (DenseFlipo  (None, 32)               2080      
 ut)                                                    

In [29]:
history = model.fit(train_x, train_y, epochs=5, validation_data=(val_x, val_y), verbose=1)

Epoch 1/5
27/27 [==============================] - 5s 49ms/step - loss: 84665.5391 - mae: 146.8717 - mse: 84601.3125 - val_loss: 30977.8418 - val_mae: 99.4654 - val_mse: 30913.6230
Epoch 2/5
27/27 [==============================] - 0s 12ms/step - loss: 32681.3105 - mae: 89.4669 - mse: 32617.0957 - val_loss: 24919.8457 - val_mae: 85.9910 - val_mse: 24855.6250
Epoch 3/5
27/27 [==============================] - 0s 12ms/step - loss: 19336.2344 - mae: 68.3276 - mse: 19272.0195 - val_loss: 12991.5586 - val_mae: 61.6578 - val_mse: 12927.3438
Epoch 4/5
27/27 [==============================] - 0s 11ms/step - loss: 17251.5898 - mae: 59.3561 - mse: 17187.3730 - val_loss: 7344.6050 - val_mae: 48.8191 - val_mse: 7280.3892
Epoch 5/5
27/27 [==============================] - 0s 12ms/step - loss: 11591.5664 - mae: 49.9376 - mse: 11527.3506 - val_loss: 8490.8447 - val_mae: 46.3905 - val_mse: 8426.6309


In [59]:
test_loss = model.evaluate(test_x, test_y)
print('Test loss:', test_loss)

2/2 [==============================] - 0s 10ms/step - loss: nan - mae: nan - mse: nan
Test loss: [nan, nan, nan]


In [28]:
model.predict(test_x[0:1])

1/1 [==============================] - 1s 859ms/step


array([[-199.70558 ,  310.19284 ,  309.2039  ,  -25.536018,  345.5504  ,
        -125.14847 ]], dtype=float32)

In [25]:
tf.keras.backend.clear_session()